# 01 - Download Sentinel-2 images of Ronti Peak (Chamoli), 2015-2021

Week 2, part A of the Slope Watch project.

What this notebook does:

1. Searches a free public catalog for Sentinel-2 scenes covering the slope.
2. Picks one or two clear scenes per autumn, 2015 to 2020, plus the last clear
   scene before the 7 February 2021 collapse.
3. Reads only a 5 km crop around the collapse scar from each chosen scene.
4. Saves the crops to `data/chamoli/` with an index file.
5. Displays them so you can check they look right.

Run the cells in order. Select the project's `.venv` as the notebook kernel first.

**Why two collections.** Sentinel-2 comes in two products. L2A is corrected for
the atmosphere and is the better one to use. L1C is the rawer version, but it
reaches further back in time. The early years are often missing from L2A, so
this notebook searches both and prefers L2A wherever it exists. Section 3 reports
which years ended up using L1C, and section 8 explains what to do about it.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from pystac_client import Client
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

# Project root: works whether the notebook runs from notebooks/ or the repo root.
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data" / "chamoli"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 5 km x 5 km box centred on the collapse scar (about 30.377 N, 79.731 E),
# located on Sentinel-2 images from 31 Jan and 10 Feb 2021 in the Copernicus Browser.
# Order: west, south, east, north (degrees).
BBOX = [79.705, 30.355, 79.757, 30.400]

CATALOG_URL = "https://earth-search.aws.element84.com/v1"
COLLECTIONS = ["sentinel-2-l2a", "sentinel-2-l1c"]   # L2A preferred, L1C as backup
BAND = "red"                    # a 10 m band; "nir" also works
DATE_RANGE = "2015-06-01/2021-02-06"   # ends the day before the collapse
SEARCH_CLOUD = 60               # wide net for the search; selection is stricter below

# Seasons to cover, one or two images each.
SEASONS = range(2015, 2021)
PREFERRED_MONTHS = [10, 11]          # least snow, similar sun angle year to year
FALLBACK_MONTHS = [9, 10, 11, 12, 1] # used only when a season has nothing better
PER_SEASON = 2

# The single most valuable image: the last clear view before the collapse.
PRE_COLLAPSE_FROM = "2020-12-01"
PRE_COLLAPSE_TO = "2021-02-06"
PRE_COLLAPSE_COUNT = 2

print("Saving to:", DATA_DIR)

## 2. Search the catalog

Both collections, a wide cloud limit, the full date range. Selection happens later.

In [ ]:
catalog = Client.open(CATALOG_URL)

items_by_id, rows = {}, []
for collection in COLLECTIONS:
    search = catalog.search(
        collections=[collection],
        bbox=BBOX,
        datetime=DATE_RANGE,
        query={"eo:cloud_cover": {"lt": SEARCH_CLOUD}},
    )
    found = list(search.items())
    print(f"{collection}: {len(found)} scenes")
    for it in found:
        items_by_id[it.id] = it
        rows.append(
            {
                "id": it.id,
                "collection": collection,
                "date": it.datetime.date(),
                "cloud": it.properties.get("eo:cloud_cover"),
            }
        )

scenes = pd.DataFrame(rows)
scenes["date"] = pd.to_datetime(scenes["date"])
scenes["year"] = scenes["date"].dt.year
scenes["month"] = scenes["date"].dt.month

# One row per date: keep L2A where both products cover the same day.
scenes["rank"] = scenes["collection"].map({c: i for i, c in enumerate(COLLECTIONS)})
scenes = (
    scenes.sort_values(["date", "rank"])
    .drop_duplicates("date", keep="first")
    .drop(columns="rank")
    .sort_values("date")
    .reset_index(drop=True)
)

# Check the band name once; BAND in cell 1 must be one of these.
print("\nAvailable assets:", sorted(items_by_id[scenes.iloc[0]["id"]].assets.keys()))
print(f"\n{len(scenes)} distinct dates")
scenes.groupby(["year", "collection"]).size().rename("scenes").reset_index()

## 3. Pick the dates

For each season, take the least cloudy scenes in October and November. If a
season has none, widen to September through January rather than lose the year.
Then add the last clear scenes before the collapse, which are needed whatever
their cloud score.

In [ ]:
def pick_season(df, season):
    """Best scenes for one autumn. A January scene belongs to the previous season."""
    in_season = df[((df["year"] == season) & (df["month"] >= 9)) |
                   ((df["year"] == season + 1) & (df["month"] == 1))]
    preferred = in_season[in_season["month"].isin(PREFERRED_MONTHS)]
    pool = preferred if len(preferred) else in_season[in_season["month"].isin(FALLBACK_MONTHS)]
    chosen = pool.sort_values("cloud").head(PER_SEASON).copy()
    chosen["why"] = f"season {season}" + ("" if len(preferred) else " (fallback months)")
    return chosen


picks = [pick_season(scenes, s) for s in SEASONS]

pre = scenes[(scenes["date"] >= PRE_COLLAPSE_FROM) & (scenes["date"] <= PRE_COLLAPSE_TO)]
pre = pre.sort_values("cloud").head(PRE_COLLAPSE_COUNT).copy()
pre["why"] = "last view before collapse"
picks.append(pre)

chosen = (
    pd.concat(picks)
    .drop_duplicates("id")
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"{len(chosen)} scenes chosen\n")
missing = [s for s in SEASONS if not any(chosen["why"].str.startswith(f"season {s}"))]
if missing:
    print("WARNING - no scene found for season(s):", missing)
if pre.empty:
    print("WARNING - no pre-collapse scene found; widen PRE_COLLAPSE_FROM in cell 1")

chosen[["date", "collection", "cloud", "why", "id"]]

Look at that table before going on.

- Is every season from 2015 to 2020 represented?
- Is there at least one scene from January or early February 2021?
- Any scene over about 20% cloud is worth checking in the Copernicus Browser:
  the figure covers the whole 100 km scene, so the slope itself may be clear
  (or may not be).

To drop or add a scene by hand, edit `CHOSEN_IDS` in the next cell.

In [ ]:
CHOSEN_IDS = list(chosen["id"])   # edit this list to override the automatic picks

chosen_items = [items_by_id[i] for i in CHOSEN_IDS]
chosen_items.sort(key=lambda it: it.datetime)
for it in chosen_items:
    print(it.datetime.date(), f"{it.properties.get('eo:cloud_cover'):6.2f}%", it.id)

## 4. Read only the crop from each scene

In [ ]:
def read_crop(item, bbox=BBOX, band=BAND):
    """Read just the bbox window of one band from a scene, without downloading the whole file."""
    with rasterio.open(item.assets[band].href) as src:
        box = transform_bounds("EPSG:4326", src.crs, *bbox)
        window = from_bounds(*box, transform=src.transform)
        return src.read(1, window=window).astype("float32")


# Try one first.
test = read_crop(chosen_items[0])
print("Crop shape:", test.shape, "| value range:", test.min(), "to", test.max())
plt.imshow(test, cmap="gray")
plt.title(str(chosen_items[0].datetime.date()))
plt.axis("off")
plt.show()

## 5. Save all the crops and an index file

Each crop is saved as `data/chamoli/YYYY-MM-DD.npy`. The index records which
collection each one came from, which matters when comparing them later.

Crops may differ by a pixel in size because of rounding. They are trimmed to a
common shape here. That is rough alignment only; measuring the real offset
between images is week 3's job.

In [ ]:
collection_of = dict(zip(chosen["id"], chosen["collection"]))

crops, rows = {}, []
for item in chosen_items:
    date = item.datetime.strftime("%Y-%m-%d")
    if date in crops:
        continue
    crop = read_crop(item)
    crops[date] = crop
    rows.append(
        {
            "date": date,
            "collection": collection_of.get(item.id, ""),
            "cloud": item.properties.get("eo:cloud_cover"),
            "id": item.id,
            "file": f"{date}.npy",
        }
    )
    print("read", date, crop.shape)

min_rows = min(c.shape[0] for c in crops.values())
min_cols = min(c.shape[1] for c in crops.values())
print(f"\ntrimming all crops to {min_rows} x {min_cols}")

for row in rows:
    crop = crops[row["date"]][:min_rows, :min_cols]
    np.save(DATA_DIR / row["file"], crop)
    row["rows"], row["cols"] = crop.shape

index = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
index.to_csv(DATA_DIR / "index.csv", index=False)
index

## 6. Display and check

Things to look for:

- The north face of Ronti Peak sits near the middle of each image.
- No crop is mostly cloud or fresh snow. Drop any that are by editing
  `CHOSEN_IDS` above, then rerun from there.
- The 2021 image should show the scar: a dark bare patch where the slope was.

In [ ]:
index = pd.read_csv(DATA_DIR / "index.csv")
n = len(index)
cols = 4
nrows = int(np.ceil(n / cols))
fig, axes = plt.subplots(nrows, cols, figsize=(16, 4 * nrows))
for ax in np.atleast_1d(axes).ravel():
    ax.axis("off")
for ax, (_, row) in zip(np.atleast_1d(axes).ravel(), index.iterrows()):
    img = np.load(DATA_DIR / row["file"])
    lo, hi = np.percentile(img, [2, 98])
    ax.imshow(np.clip((img - lo) / (hi - lo + 1e-9), 0, 1), cmap="gray")
    ax.set_title(f"{row['date']}  {row['collection'][-3:].upper()}  cloud {row['cloud']:.0f}%", fontsize=9)
plt.tight_layout()
plt.show()

print("Shapes:", sorted(set(zip(index["rows"], index["cols"]))))
print("\nCoverage by year:")
print(pd.to_datetime(index["date"]).dt.year.value_counts().sort_index().to_string())

## 7. Compare the first and last image

Flip between these two. Over five years the slope moves only a fraction of a
pixel a year, so nothing should look obviously different unless the later image
is after the collapse.

In [ ]:
first, last = index.iloc[0], index.iloc[-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, row in zip(axes, [first, last]):
    img = np.load(DATA_DIR / row["file"])
    lo, hi = np.percentile(img, [2, 98])
    ax.imshow(np.clip((img - lo) / (hi - lo + 1e-9), 0, 1), cmap="gray")
    ax.set_title(f"{row['date']}  ({row['collection'][-3:].upper()})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Locate the scar exactly, using an image from after the collapse

Everything above is from before 7 February 2021, so the slope still looks intact
and the collapse source can only be guessed at. One image from after the collapse
settles it: the scar shows up as a dark bare patch where snow and rock used to be.

**Take the earliest clear image, not the clearest one.** The scar is exposed rock.
Each snowfall covers more of it, so by March the slope is white again and the scar
has disappeared. The 10 February scene, three days after the event, is the one that
shows it plainly.

Subtracting a before image from an after image makes it stand out, and the result
gives exact pixel bounds for the block that fell. Those bounds are the ground truth
the tracker is measured against in week 3.

**This image is kept out of the main set.** It goes in `data/chamoli/after/` and is
not added to `index.csv`, because mixing a post-collapse image into the time series
would corrupt every measurement.

In [ ]:
AFTER_DIR = DATA_DIR / "after"
AFTER_DIR.mkdir(exist_ok=True)

# Pick the EARLIEST clear scene after the collapse, not the clearest one.
# The scar is bare rock. Every snowfall hides it a little more, so a very clear
# image from weeks later shows nothing: by then the whole slope is white again.
AFTER_MAX_CLOUD = 25

after_search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2021-02-08/2021-03-20",
    query={"eo:cloud_cover": {"lt": 60}},
)
after_scenes = sorted(after_search.items(), key=lambda it: it.datetime)

print("candidates, oldest first:")
for it in after_scenes[:10]:
    cloud = it.properties.get("eo:cloud_cover")
    mark = "  <-- first clear one" if cloud < AFTER_MAX_CLOUD else ""
    print(f"  {it.datetime.date()}  {cloud:6.2f}%{mark}")

clear = [it for it in after_scenes if it.properties.get("eo:cloud_cover", 100) < AFTER_MAX_CLOUD]
if not clear:
    raise SystemExit("no clear scene found - raise AFTER_MAX_CLOUD")

after_item = clear[0]                   # earliest clear one
after_date = after_item.datetime.strftime("%Y-%m-%d")
after_img = read_crop(after_item)[:min_rows, :min_cols]
np.save(AFTER_DIR / f"{after_date}.npy", after_img)
days = (after_item.datetime.date() - pd.Timestamp("2021-02-07").date()).days
print(f"\nusing {after_date}, {days} days after the collapse")
print("saved", AFTER_DIR / f"{after_date}.npy", after_img.shape)

### Before and after

The scar should be obvious: a dark patch on the slope, with a trail of debris
running down the valley away from it.

In [ ]:
before_row = index.iloc[-1]             # last pre-collapse image
before_img = np.load(DATA_DIR / before_row["file"])


def stretch(a):
    lo, hi = np.nanpercentile(a, [2, 98])
    return np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)


fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(stretch(before_img), cmap="gray")
axes[0].set_title(f"before: {before_row['date']}")
axes[1].imshow(stretch(after_img), cmap="gray")
axes[1].set_title(f"after: {after_date}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Find the scar automatically

Each image is scaled to the same brightness range first, because the sun angle and
snow cover differ between the two dates. The scar is then the largest patch that got
markedly darker.

The debris trail also darkens, and it runs well beyond the source area, so the patch
found here may be larger than the block itself. Look at the result before trusting it.

In [ ]:
from scipy import ndimage


def normalise(a):
    """Zero mean, unit spread, so two dates can be compared despite different lighting."""
    a = a.astype("float64")
    return (a - np.nanmean(a)) / (np.nanstd(a) + 1e-9)


diff = normalise(after_img) - normalise(before_img)

DARKEN_PERCENTILE = 2       # how much of the image counts as "much darker"
threshold = np.nanpercentile(diff, DARKEN_PERCENTILE)
darker = ndimage.binary_opening(diff < threshold, np.ones((3, 3)))

labels, count = ndimage.label(darker)
if count:
    sizes = ndimage.sum(darker, labels, range(1, count + 1))
    biggest = int(np.argmax(sizes)) + 1
    rows_idx, cols_idx = np.where(labels == biggest)
    r0, r1, c0, c1 = rows_idx.min(), rows_idx.max(), cols_idx.min(), cols_idx.max()
    print(f"largest darkened patch: rows {r0}-{r1}, cols {c0}-{c1}")
    print(f"size: {int(sizes[biggest - 1])} pixels = {int(sizes[biggest - 1]) * 100 / 1e6:.3f} km2")
    print(f"centre: row {(r0 + r1) // 2}, col {(c0 + c1) // 2}")
    print(f"\nSCAR_BOX = ({r0}, {r1}, {c0}, {c1})   # for the tracker in week 3")
else:
    print("nothing found - try raising DARKEN_PERCENTILE")

### Check it by eye

In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(stretch(before_img), cmap="gray")
axes[0].set_title(f"before: {before_row['date']}")
axes[1].imshow(stretch(after_img), cmap="gray")
axes[1].set_title(f"after: {after_date}")
axes[2].imshow(diff, cmap="RdBu", vmin=-3, vmax=3)
axes[2].set_title("difference (blue = darker after)")

if count:
    for ax in axes:
        ax.add_patch(Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, ec="#ff3b30", lw=2))
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

Does the red box sit on the slope that failed, rather than on a shifting snow patch
or a shadow? If not, raise or lower `DARKEN_PERCENTILE` and run again, or read the
bounds off the difference image by eye.

Write the final `SCAR_BOX` into your notes. Week 3 uses it to decide which patch of
the image the tracker should follow.

## 9. If some years came from L1C

L1C and L2A measure brightness differently: L1C is light at the top of the
atmosphere, L2A is light at the ground after the haze is removed. So the same
rock can have a different value in the two products.

For tracking movement this matters less than it sounds, because the method
compares patterns rather than absolute brightness, and each image is scaled
before use. But it is a real difference and belongs in the report. Two options:

- **Mixed, and say so.** Keep L2A where available, note in the index which years
  are L1C, and check in week 3 that the tracker gives sensible results across a
  boundary between the two.
- **All L1C.** Set `COLLECTIONS = ["sentinel-2-l1c"]` in cell 1 and rerun
  everything. Consistency across years, at the cost of atmospheric correction.

Try the first. If week 3 shows odd jumps at the product boundary, switch.

---
Contains modified Copernicus Sentinel data, accessed through the Earth Search catalog on AWS Open Data.

Next: `02_testbed.ipynb` (shift a patch by a known amount, add noise and fake clouds).